
<img src="../img/GTK_Logo_Social_Icon.jpg" width=175 align="right" />

# Worksheet 11.1 AI: Retrieval Augmented Generation (RAG) - Answers

This notebook shows how to build a semantic search engine using **RAG**. 

The task is to build a model that will be able to take in a plain language query and find the most relevant documents to answer this query. 

#### Data: 
[https://www.kaggle.com/datasets/manavkhambhayata/cve-2024-database-exploits-cvss-os](https://www.kaggle.com/datasets/manavkhambhayata/cve-2024-database-exploits-cvss-os)

The data comes from the National Vulnerability Database (NVD), a government-managed repository of cybersecurity vulnerabilities. It provides detailed information on security issues, including severity scores and affected systems.

The dataset was extracted using the NVD API and processed with Python. It includes vulnerabilities published between January 1, 2024, and January 15, 2024, with key details such as CVE ID, description, CVSS score, attack vector, and affected operating systems.

#### The two halves of RAG

A RAG pipeline has two model-powered steps, and they use **different** models:

1. **Retrieval** — embed the documents and the query so we can find the closest matches. This needs an *embedding* model. Anthropic does **not** offer an embeddings API (Claude is generation-only), so we run a local open-source **sentence-transformer** through `langchain-huggingface`. It runs offline and needs no API key.
2. **Generation** — hand the retrieved documents to a chat model and have it answer the question *grounded on those documents*. Here we use **Anthropic's Claude** (`claude-opus-4-8`) via `langchain-anthropic`.

#### Libraries:

- [langchain](https://www.langchain.com/langchain) — this notebook uses the langchain v1 SDK
- [langchain-huggingface](https://pypi.org/project/langchain-huggingface/) — local sentence-transformer embeddings for the retrieval half
- [langchain-anthropic](https://pypi.org/project/langchain-anthropic/) — Claude chat models for the generation half
- [python-dotenv](https://pypi.org/project/python-dotenv/) — loads your `ANTHROPIC_API_KEY` from a `.env` file so it never gets hardcoded

> Put `ANTHROPIC_API_KEY=sk-ant-...` in a `.env` file in the project root (it's already git-ignored). All packages are pre-installed in the bootcamp environment.

In [3]:
# Load Libraries - Make sure to run this cell!
import warnings

# Silence the langchain-community "sunset" DeprecationWarning. It fires at IMPORT time,
# so this filter must come BEFORE the langchain imports below (not after them).
warnings.filterwarnings("ignore")

from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from dotenv import load_dotenv
load_dotenv()  # reads ANTHROPIC_API_KEY from your .env file in the project root

True

## Load data
Langchain has a variety of nice modules that help you load different formats of documents. [document loaders](https://python.langchain.com/docs/how_to/#document-loaders)

In [4]:
DATA_HOME = '../data/'

filename_cves = 'nvd_vulnerabilities_with_os.csv'

In [5]:
loader = CSVLoader(
    file_path=DATA_HOME + filename_cves, 
    source_column='CVE ID',
    # Promote structured columns into each document's metadata. The text still gets
    # embedded, but keeping CVSS Score + Affected OS in metadata lets us do a hybrid
    # search later (semantic similarity + a structured filter on severity).
    metadata_columns=['CVSS Score', 'Affected OS'],
    csv_args={
        "delimiter": ",",
        "quotechar": '"',
    },
     encoding="UTF-8"
)

docs_raw = loader.load()
for record in docs_raw[:2]:
    print(record)
    print('----------')

page_content='CVE ID: CVE-2024-21732
Description: FlyCms through abbaa5a allows XSS via the permission management feature.
Attack Vector: CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N' metadata={'source': 'CVE-2024-21732', 'row': 0, 'CVSS Score': '6.1', 'Affected OS': 'N/A'}
----------
page_content='CVE ID: CVE-2023-5877
Description: The affiliate-toolkit WordPress plugin before 3.4.3 lacks authorization and authentication for requests to it's affiliate-toolkit-starter/tools/atkp_imagereceiver.php endpoint, allowing unauthenticated visitors to make requests to arbitrary URL's, including RFC1918 private addresses, leading to a Server Side Request Forgery (SSRF) issue.
Attack Vector: CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H' metadata={'source': 'CVE-2023-5877', 'row': 1, 'CVSS Score': '9.8', 'Affected OS': 'N/A'}
----------


> **A note on chunking.** Each CVE record here is short — a description plus a few fields — so one row = one document = one embedding works fine. On real data (PDFs, long reports, web pages) a single document is often longer than the embedding model's input limit, and cramming a whole page into one vector blurs the meaning of everything in it. In that case you'd first split documents into overlapping chunks with a [`RecursiveCharacterTextSplitter`](https://python.langchain.com/docs/how_to/recursive_text_splitter/) and embed each chunk. We skip it here because our records already fit — but it's the #1 thing you'll add when you move to bigger documents.

## Create Embeddings

The retrieval half of RAG needs an **embedding model**. LangChain has integration packages for many model servers (OpenAI, Azure, AWS, Google, HuggingFace, etc.). [https://python.langchain.com/docs/tutorials/retrievers/#embeddings](https://python.langchain.com/docs/tutorials/retrievers/#embeddings)

**Note on Anthropic:** Claude is a generation-only model server — Anthropic does not expose an embeddings endpoint. For embeddings we use a local open-source **sentence-transformer** via `langchain-huggingface`, which runs entirely offline (no API key, works air-gapped). Claude comes back for the *generation* step at the end.

You have some decisions to make at this point: 

**Decision 1**: Select model server (or this is where you could access a downloaded model if working in an airgapped environment)

**Decision 2**: Select the specific model to use to generate your embeddings. Since we are doing a semantic search engine, you will want something that is good at that task like a sentence-transformer. This is where you can end up with something that is great for your task, or something that is terrible. Once you have your pipeline, you will want to try out some different models to see how the results change. Things to consider:

 - size (larger might be better)
 - tokenizer (do you need multi-language capabilities?, this is where you would ensure it has seen your languages)
 - origin (use something that comes from a reputable source, be wary of things that have been fine-tuned by random people in places like huggingface. go for the ones from Microsoft, Facebook etc)
 - content for training (do you need to know this? if so, you'll want an open source model)
 - input length (models can only take in a specific number of tokens, older, smaller models have a smaller amount of tokens they can take in which affects your ability to embed large/long texts


In [ ]:
# Local sentence-transformer — downloads once, then runs offline. No API key needed.
# Try swapping in "sentence-transformers/all-mpnet-base-v2" (larger, 768-dim) to see results change.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Try out your new embedding model on a few of our documents. The **embed_query** method is typically used to embed a single sentence, like we do for an incoming query, which is why it's useful here to just see what it does on one document. 

In [7]:
vector_1 = embeddings.embed_query(docs_raw[0].page_content)
vector_2 = embeddings.embed_query(docs_raw[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 384

[-0.07591865956783295, 0.07137496024370193, -0.15056884288787842, -0.009640743024647236, 0.07238541543483734, -0.005045372061431408, 0.1056818962097168, 0.009404338896274567, -0.027718989178538322, 0.05282103270292282]


## Storage
Now that we know how to embed our documents, we will need to store these embeddings in a database. There are many options for doing this, but the most efficient way to store embeddings is in a **vector database**. These have been optimized to store and retrieve these kinds of embeddings, so when you can use them, you should. You can also use more traditional things like MongoBD or Elasticsearch with specific field types for storing a dense vector. This is useful if you need to store a lot of metadata or continue to have the option do to a keyword search in addition to the semantic search (this is common).

In [ ]:
vector_store = InMemoryVectorStore(embeddings)

In [ ]:
in_memory = vector_store.add_documents(documents=docs_raw)

## Semantic Search
we will use similarity search (which uses cosine similarity) to find the documents that are the most similar to our query. Then look at the 5 most relevant.

This time we also print the **relevance score** for each hit. Watch the scores drop for the later results — and try an off-topic query (e.g. *"how do I bake bread?"*): the store *still* returns its `k` nearest documents, but the scores will be low. That's the intuition behind a **relevance threshold**: in production you'd discard hits below some score so a bad query returns "no relevant results" instead of confidently wrong ones.

In [ ]:
# similarity_search_with_score returns a relevance score alongside each hit, so you
# can see HOW good the matches are — not just their order. For InMemoryVectorStore
# this score is cosine similarity: higher = more similar (1.0 = identical direction).
results_scored = vector_store.similarity_search_with_score(
    "Chrome vulnerabilities to heap corruption in May.", k=5
)
for doc, score in results_scored:
    print(f"[score {score:.3f}] {doc.metadata['source']}")
    print(doc.page_content)
    print('-----')

## Semantic Search 2
Return documents based on similarity to an embedded query. Now we want to embed out query (USING THE SAME EMBEDDING MODEL AS THE DOCS). Then compare our query's vector to the database vectors and get the ones with the smallest distance between the 2. There are multiple ways to calculate the *distance* between 2 vectors, but the most popular for this task is cosine similarity.

In [ ]:
embedding = embeddings.embed_query("Which phone OS is the most vulnerable to shell injection?")

results = vector_store.similarity_search_by_vector(embedding, k=5)
for doc in results[0:5]:
    print(doc)
    print('-----')

## Hybrid search: semantic + metadata filter
Real vulnerability triage rarely wants *just* the closest match — it wants the closest match **that also meets a structured criterion**. "Find me RCE-like CVEs, but only High/Critical severity." Because we promoted `CVSS Score` into each document's metadata when we loaded the data, we can combine semantic similarity with a metadata `filter`.

The `filter` is just a function that takes a `Document` and returns `True`/`False`. Only documents that pass are eligible to be returned.

In [12]:
def high_severity(doc: Document) -> bool:
    """Keep only CVEs with CVSS >= 8.0 (High/Critical)."""
    try:
        return float(doc.metadata.get("CVSS Score", "0")) >= 8.0
    except ValueError:
        return False  # some rows have a non-numeric / missing score

results = vector_store.similarity_search(
    "remote code execution", k=5, filter=high_severity
)
for doc in results:
    print(f"CVSS {doc.metadata['CVSS Score']} - {doc.metadata['source']}")
    print(doc.page_content.splitlines()[1][:100])
    print('-----')

CVSS 9.8 - CVE-2024-0552
Description: Intumit inc. SmartRobot's web framwork has a remote code execution vulnerability. An un
-----
CVSS 8.8 - CVE-2023-51066
Description: An authenticated remote code execution vulnerability in QStar Archive Solutions Release
-----
CVSS 9.8 - CVE-2024-22086
Description: handle_request in http.c in cherry through 4b877df has an sscanf stack-based buffer ove
-----
CVSS 8.8 - CVE-2024-21318
Description: Microsoft SharePoint Server Remote Code Execution Vulnerability
-----
CVSS 9.8 - CVE-2024-22087
Description: route in main.c in Pico HTTP Server in C through f3b69a6 has an sprintf stack-based buf
-----


# Generated Answers
This RAG architecture is usually part 1 of a chatbot (and many other products). Retrieval gets us the relevant source documents for a user's query; now we take those top X docs and give them to a **generative model** — here, Anthropic's **Claude** — to generate an answer to the question *grounded on those documents*.

We wire this together with the langchain [Runnable](https://python.langchain.com/docs/concepts/lcel/) pipe syntax: `prompt | llm`. The prompt template injects the retrieved CVE docs as context, and the `system` instruction tells Claude to answer **only** from them — the guardrail that keeps a RAG bot from making things up.

In [ ]:
# 1. Retrieve the source docs for a question
question = "Which phone OS is the most vulnerable to shell injection?"
results = vector_store.similarity_search(question, k=5)

# 2. Build the grounded prompt. The {context} slot gets the retrieved CVE docs.
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a cybersecurity assistant. Answer the user's question using ONLY the "
     "CVE documents below. If the answer is not in them, say you don't have enough "
     "information. Cite the relevant CVE IDs.\n\n{context}"),
    ("human", "{question}"),
])

# 3. The generation model: Anthropic's Claude via langchain-anthropic.
#    Reads ANTHROPIC_API_KEY from the environment (loaded from .env above).
llm = ChatAnthropic(model="claude-opus-4-8", max_tokens=1024)

# 4. Wire prompt -> llm with the Runnable pipe and invoke it.
context = "\n\n".join(doc.page_content for doc in results)
chain = prompt | llm
answer = chain.invoke({"context": context, "question": question})

print(answer.content)

In [ ]:
# Show the sources Claude was grounded on — so a user can verify the answer.
print("Sources used:")
for doc in results:
    print(f"  {doc.metadata['source']} - {doc.page_content.splitlines()[1][:80]}")

## Streaming the answer
`.invoke()` waits for the whole response before returning. For a chatbot UI you usually want tokens to appear as they're generated — swap `.invoke()` for `.stream()` and print each chunk as it arrives. Same chain, same prompt; only the call changes.

In [15]:
for chunk in chain.stream({"context": context, "question": question}):
    print(chunk.content, end="", flush=True)

Based on the CVE documents provided, the vulnerable phone OS is **Android**, specifically as implemented in **PAX Android-based POS devices**.

Two relevant CVEs affect these devices running **PayDroid_8.1.0_Sagittarius_V11.1.50_20230614 or earlier**:

- **CVE-2023-42136**: Allows execution of arbitrary commands with system account privilege via shell injection starting with a specific word.
- **CVE-2023-42137**: Allows command execution with high privileges by using malicious symlinks.

Both vulnerabilities require the attacker to have shell access to the device, and both carry a high severity rating (CVSS 3.1 vector: AV:L/AC:L/PR:L/UI:N/S:U/C:H/I:H/A:H).

The other CVEs in the documents (CVE-2023-47560, CVE-2023-39294, CVE-2023-41289) relate to QNAP products and operating systems, not phone OSes. Note also that only Android is referenced in the provided documents — I don't have information comparing it against other phone operating systems, so I can only confirm Android (PAX POS devi